<a href="https://colab.research.google.com/github/Muhammad-Mehdi-kazmi/AI-LABS/blob/main/AI_LAB_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

class VacuumEnvironment:
    """Two-square world: locations 'A' and 'B', each Clean (0) or Dirty (1)."""
    def __init__(self, status=None, location=None):
        self.status = status or {'A': random.randint(0, 1), 'B': random.randint(0, 1)}
        self.agent_location = location or random.choice(['A', 'B'])

    def percept(self):
        return (self.agent_location, self.status[self.agent_location])

    def execute(self, action, strict=True):
        """Apply action, return the reward for that step.
        strict=True  -> +10 only if the square was actually dirty
        strict=False -> original rule: every Suck gives +10"""
        if action == 'Suck':
            was_dirty = self.status[self.agent_location]
            self.status[self.agent_location] = 0
            if strict:
                return 10 if was_dirty else -1
            return 10
        elif action == 'Right':
            self.agent_location = 'B'
        elif action == 'Left':
            self.agent_location = 'A'
        return -1  # cost of moving


def reflex_vacuum_agent(percept):
    """Condition-action rules: decide using ONLY the current percept."""
    location, dirty = percept
    if dirty:
        return 'Suck'
    elif location == 'A':
        return 'Right'
    else:
        return 'Left'


def random_vacuum_agent(percept):
    """Picks uniformly from {Left, Right, Suck}; ignores the percept."""
    return random.choice(['Left', 'Right', 'Suck'])


def run(agent, status, location, steps=20, strict=True):
    env = VacuumEnvironment(dict(status), location)  # same start for both agents
    score = 0
    for _ in range(steps):
        score += env.execute(agent(env.percept()), strict)
    return score


def compare(strict, steps=20, trials=10):
    starts = []
    for _ in range(trials):
        e = VacuumEnvironment()
        starts.append((dict(e.status), e.agent_location))

    print(f"{'Trial':<6}{'Initial state':<26}{'Reflex':>8}{'Random':>8}")
    reflex, rand = [], []
    for i, (st, loc) in enumerate(starts, 1):
        r = run(reflex_vacuum_agent, st, loc, steps, strict)
        n = run(random_vacuum_agent, st, loc, steps, strict)
        reflex.append(r); rand.append(n)
        print(f"{i:<6}{str(st) + ' @' + loc:<26}{r:>8}{n:>8}")
    print(f"Average reflex score: {sum(reflex) / trials:.1f}")
    print(f"Average random score: {sum(rand) / trials:.1f}\n")


random.seed(7)
print("=== Original scoring (every Suck = +10) ===")
compare(strict=False)

random.seed(7)
print("=== Fixed scoring (+10 only when square was dirty) ===")
compare(strict=True)

=== Original scoring (every Suck = +10) ===
Trial Initial state               Reflex  Random
1     {'A': 1, 'B': 0} @B             -9      90
2     {'A': 0, 'B': 0} @A            -20      57
3     {'A': 1, 'B': 0} @A             -9      46
4     {'A': 0, 'B': 0} @B            -20      79
5     {'A': 1, 'B': 0} @A             -9      24
6     {'A': 0, 'B': 1} @A             -9      24
7     {'A': 0, 'B': 0} @A            -20      46
8     {'A': 1, 'B': 0} @A             -9      68
9     {'A': 0, 'B': 0} @B            -20      35
10    {'A': 1, 'B': 0} @A             -9      24
Average reflex score: -13.4
Average random score: 49.3

=== Fixed scoring (+10 only when square was dirty) ===
Trial Initial state               Reflex  Random
1     {'A': 1, 'B': 0} @B             -9      -9
2     {'A': 0, 'B': 0} @A            -20     -20
3     {'A': 1, 'B': 0} @A             -9      -9
4     {'A': 0, 'B': 0} @B            -20     -20
5     {'A': 1, 'B': 0} @A             -9      -9
6     {'A': 

In [2]:
import random
from itertools import product

class VacuumEnvironment:
    """Two-square world. Moves cost `move_cost`, NoOp is free, Suck pays +10 only if dirty."""
    def __init__(self, status=None, location=None, move_cost=1):
        self.status = status or {'A': random.randint(0, 1), 'B': random.randint(0, 1)}
        self.agent_location = location or random.choice(['A', 'B'])
        self.move_cost = move_cost

    def percept(self):
        return (self.agent_location, self.status[self.agent_location])

    def execute(self, action):
        if action == 'Suck':
            was_dirty = self.status[self.agent_location]
            self.status[self.agent_location] = 0
            return 10 if was_dirty else -1      # wasted suck costs 1
        if action == 'Right':
            self.agent_location = 'B'; return -self.move_cost
        if action == 'Left':
            self.agent_location = 'A'; return -self.move_cost
        return 0                                # NoOp: stopping is free


def reflex_vacuum_agent(percept):
    """Stateless: never stops, keeps patrolling forever."""
    location, dirty = percept
    if dirty:
        return 'Suck'
    return 'Right' if location == 'A' else 'Left'


class ModelBasedVacuumAgent:
    """Keeps an internal model: what it believes about each square.
    None = unknown, 1 = dirty, 0 = clean (cleaned or seen clean)."""
    def __init__(self):
        self.model = {'A': None, 'B': None}

    def __call__(self, percept):
        location, dirty = percept
        self.model[location] = dirty            # 1. update model from percept

        if dirty:                               # 2. choose action
            self.model[location] = 0            #    remember: this square will be clean
            return 'Suck'
        if self.model['A'] == 0 and self.model['B'] == 0:
            return 'NoOp'                       #    both known clean -> stop
        return 'Right' if location == 'A' else 'Left'   # go check the other square


def run(make_agent, status, location, steps=20, move_cost=1):
    env = VacuumEnvironment(dict(status), location, move_cost)
    agent = make_agent()                        # fresh agent (fresh memory) every run
    return sum(env.execute(agent(env.percept())) for _ in range(steps))


reflex = lambda: reflex_vacuum_agent
model  = lambda: ModelBasedVacuumAgent()

# ---- Experiment 1: 10 random initial states, moves cost 1 ----
random.seed(7)
starts = []
for _ in range(10):
    e = VacuumEnvironment()
    starts.append((dict(e.status), e.agent_location))

print("Experiment 1: 10 random starts, 20 steps, move cost = 1")
print(f"{'Trial':<6}{'Initial state':<26}{'Reflex':>8}{'Model':>8}")
rs, ms = [], []
for i, (st, loc) in enumerate(starts, 1):
    r, m = run(reflex, st, loc), run(model, st, loc)
    rs.append(r); ms.append(m)
    print(f"{i:<6}{str(st) + ' @' + loc:<26}{r:>8}{m:>8}")
print(f"Average reflex: {sum(rs)/10:.1f} | Average model-based: {sum(ms)/10:.1f}\n")

# ---- Experiment 2: ALL 8 possible initial states (exhaustive proof) ----
print("Experiment 2: all 8 initial states, move cost = 1")
print(f"{'A':<3}{'B':<3}{'Loc':<5}{'Reflex':>8}{'Model':>8}")
all_ok = True
for a, b, loc in product([0, 1], [0, 1], ['A', 'B']):
    st = {'A': a, 'B': b}
    r, m = run(reflex, st, loc), run(model, st, loc)
    all_ok &= m > r
    print(f"{a:<3}{b:<3}{loc:<5}{r:>8}{m:>8}")
print("Model-based strictly better in every state:", all_ok, "\n")

# ---- Experiment 3: effect of the move penalty ----
print("Experiment 3: average over all 8 states vs. move cost")
print(f"{'Move cost':<10}{'Reflex':>8}{'Model':>8}")
for mc in [0, 1, 2]:
    states = list(product([0, 1], [0, 1], ['A', 'B']))
    r = sum(run(reflex, {'A': a, 'B': b}, l, move_cost=mc) for a, b, l in states) / 8
    m = sum(run(model, {'A': a, 'B': b}, l, move_cost=mc) for a, b, l in states) / 8
    print(f"{mc:<10}{r:>8.1f}{m:>8.1f}")

Experiment 1: 10 random starts, 20 steps, move cost = 1
Trial Initial state               Reflex   Model
1     {'A': 1, 'B': 0} @B             -9       9
2     {'A': 0, 'B': 0} @A            -20      -1
3     {'A': 1, 'B': 0} @A             -9       9
4     {'A': 0, 'B': 0} @B            -20      -1
5     {'A': 1, 'B': 0} @A             -9       9
6     {'A': 0, 'B': 1} @A             -9       9
7     {'A': 0, 'B': 0} @A            -20      -1
8     {'A': 1, 'B': 0} @A             -9       9
9     {'A': 0, 'B': 0} @B            -20      -1
10    {'A': 1, 'B': 0} @A             -9       9
Average reflex: -13.4 | Average model-based: 5.0

Experiment 2: all 8 initial states, move cost = 1
A  B  Loc    Reflex   Model
0  0  A         -20      -1
0  0  B         -20      -1
0  1  A          -9       9
0  1  B          -9       9
1  0  A          -9       9
1  0  B          -9       9
1  1  A           2      19
1  1  B           2      19
Model-based strictly better in every state: True 

Ex

In [3]:
import random
from collections import deque

N = 3
MOVES = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}


class GridVacuumEnvironment:
    """N x N grid of rooms. Each room is Clean (0) or Dirty (1).
    Rewards: +10 for cleaning a dirty room, -1 per move, -1 for a wasted Suck, 0 for NoOp."""
    def __init__(self, n=N, dirt_prob=0.5):
        self.n = n
        self.status = {(r, c): int(random.random() < dirt_prob)
                       for r in range(n) for c in range(n)}
        self.location = (random.randrange(n), random.randrange(n))
        self.moves = self.cleaned = self.score = self.steps = 0

    def percept(self):
        return (self.location, self.status[self.location])   # sees ONLY its own room

    def execute(self, action):
        self.steps += 1
        if action == 'Suck':
            if self.status[self.location]:
                self.status[self.location] = 0
                self.cleaned += 1
                self.score += 10
            else:
                self.score -= 1
        elif action in MOVES:
            dr, dc = MOVES[action]
            r, c = self.location[0] + dr, self.location[1] + dc
            if 0 <= r < self.n and 0 <= c < self.n:
                self.location = (r, c)
            self.moves += 1
            self.score -= 1
        # 'NoOp' costs nothing

    def grid_str(self, mark_agent=False):
        rows = []
        for r in range(self.n):
            row = []
            for c in range(self.n):
                ch = 'D' if self.status[(r, c)] else '.'
                if mark_agent and (r, c) == self.location:
                    ch = '[' + ch + ']'
                else:
                    ch = ' ' + ch + ' '
                row.append(ch)
            rows.append(''.join(row))
        return '\n'.join(rows)


class ModelBasedGridAgent:
    """Model-based reflex agent.
    Internal model: for every room -> None (never visited), 0 (clean), 1 (dirty).
    Rules:  1) update model from percept
            2) dirty room            -> Suck
            3) every room known      -> NoOp  (stop)
            4) otherwise             -> step toward the nearest unvisited room."""
    def __init__(self, n=N):
        self.n = n
        self.model = {(r, c): None for r in range(n) for c in range(n)}

    def __call__(self, percept):
        loc, dirty = percept
        self.model[loc] = dirty                      # 1. update model
        if dirty:                                    # 2. clean it
            self.model[loc] = 0
            return 'Suck'
        if all(v is not None for v in self.model.values()):
            return 'NoOp'                            # 3. nothing left to do
        return self._step_toward_nearest_unvisited(loc)   # 4. explore

    def _step_toward_nearest_unvisited(self, start):
        queue, seen = deque([(start, None)]), {start}
        while queue:                                 # BFS over the grid
            (r, c), first = queue.popleft()
            if self.model[(r, c)] is None:
                return first
            for name, (dr, dc) in MOVES.items():
                nxt = (r + dr, c + dc)
                if 0 <= nxt[0] < self.n and 0 <= nxt[1] < self.n and nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, first or name))


def min_moves_to_visit_all(start, n=N):
    """Lower bound: fewest moves of ANY strategy that visits every room (BFS over (pos, visited))."""
    full = (1 << (n * n)) - 1
    bit = lambda p: 1 << (p[0] * n + p[1])
    queue, seen = deque([(start, bit(start), 0)]), {(start, bit(start))}
    while queue:
        pos, mask, d = queue.popleft()
        if mask == full:
            return d
        for dr, dc in MOVES.values():
            nxt = (pos[0] + dr, pos[1] + dc)
            if 0 <= nxt[0] < n and 0 <= nxt[1] < n:
                state = (nxt, mask | bit(nxt))
                if state not in seen:
                    seen.add(state)
                    queue.append((*state, d + 1))


def run(env, max_steps=100, trace=False):
    agent = ModelBasedGridAgent(env.n)
    for _ in range(max_steps):
        p = env.percept()
        a = agent(p)
        if trace:
            print(f"  step {env.steps + 1:2d}: at {p[0]} dirty={p[1]} -> {a}")
        if a == 'NoOp':
            break
        env.execute(a)
    return agent


# ------------------------- one detailed run -------------------------
random.seed(3)
env = GridVacuumEnvironment()
start, dirty_rooms = env.location, sum(env.status.values())
print("Initial world (D = dirty, [ ] = agent):")
print(env.grid_str(mark_agent=True))
print(f"Agent starts at {start}; {dirty_rooms} dirty rooms\n")
print("Action trace:")
agent = run(env, trace=True)
print("\nFinal world:")
print(env.grid_str(mark_agent=True))
print(f"\nRooms visited : {sum(v is not None for v in agent.model.values())} / {N*N}")
print(f"Rooms cleaned : {env.cleaned} / {dirty_rooms} dirty")
print(f"Moves         : {env.moves} (minimum possible: {min_moves_to_visit_all(start)})")
print(f"Score         : {env.score}  (= 10*{env.cleaned} - {env.moves})\n")

# ------------------------- 10 random worlds -------------------------
random.seed(7)
print("10 random worlds:")
print(f"{'#':<3}{'start':<8}{'dirty':>6}{'cleaned':>9}{'moves':>7}{'min':>5}{'score':>7}  all clean?")
tot = dict(cleaned=0, moves=0, score=0)
for i in range(1, 11):
    e = GridVacuumEnvironment()
    s, d = e.location, sum(e.status.values())
    run(e)
    ok = not any(e.status.values())
    print(f"{i:<3}{str(s):<8}{d:>6}{e.cleaned:>9}{e.moves:>7}{min_moves_to_visit_all(s):>5}{e.score:>7}  {ok}")
    tot['cleaned'] += e.cleaned; tot['moves'] += e.moves; tot['score'] += e.score
print(f"Averages: cleaned={tot['cleaned']/10:.1f}, moves={tot['moves']/10:.1f}, score={tot['score']/10:.1f}")

# ------------------------- large sample -------------------------
random.seed(1)
T = 5000
m = c = s = 0
fully = 0
for _ in range(T):
    e = GridVacuumEnvironment()
    run(e)
    m += e.moves; c += e.cleaned; s += e.score
    fully += not any(e.status.values())
print(f"\nOver {T} random worlds: moves={m/T:.2f}, cleaned={c/T:.2f}, "
      f"score={s/T:.2f}, fully cleaned in {fully/T:.0%}")

Initial world (D = dirty, [ ] = agent):
[D] .  D 
 .  .  D 
 D  .  D 
Agent starts at (0, 0); 5 dirty rooms

Action trace:
  step  1: at (0, 0) dirty=1 -> Suck
  step  2: at (0, 0) dirty=0 -> Down
  step  3: at (1, 0) dirty=0 -> Down
  step  4: at (2, 0) dirty=1 -> Suck
  step  5: at (2, 0) dirty=0 -> Right
  step  6: at (2, 1) dirty=0 -> Up
  step  7: at (1, 1) dirty=0 -> Up
  step  8: at (0, 1) dirty=0 -> Right
  step  9: at (0, 2) dirty=1 -> Suck
  step 10: at (0, 2) dirty=0 -> Down
  step 11: at (1, 2) dirty=1 -> Suck
  step 12: at (1, 2) dirty=0 -> Down
  step 13: at (2, 2) dirty=1 -> Suck
  step 14: at (2, 2) dirty=0 -> NoOp

Final world:
 .  .  . 
 .  .  . 
 .  . [.]

Rooms visited : 9 / 9
Rooms cleaned : 5 / 5 dirty
Moves         : 8 (minimum possible: 8)
Score         : 42  (= 10*5 - 8)

10 random worlds:
#  start    dirty  cleaned  moves  min  score  all clean?
1  (1, 1)       6        6      8    8     52  True
2  (1, 0)       5        5      9    9     41  True
3  (2, 2)   